In [1]:
import torch
if torch.cuda.is_available():    
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

In [2]:
import pandas as pd
import re
import os

data = pd.read_table("../input/project-2-data/EmotionTrain.tsv")
data.columns = ['Phrase', 'Emotion', 'Garbage']
data.drop(['Garbage'], axis=1, inplace=True)

emos = ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 
'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism',
'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']

emotionDict = {}
for index, emo in enumerate(emos):
    emotionDict[str(index)] = emo
    
ekmanDictMo = {
    "anger": ["anger", "annoyance", "disapproval"],
    "disgust": ["disgust"],
    "fear": ["fear", "nervousness"],
    "joy": ["joy", "amusement", "approval", "excitement", "gratitude",  "love", "optimism", "relief", "pride", "admiration", "desire", "caring"],
    "sadness": ["sadness", "disappointment", "embarrassment", "grief",  "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral": ["neutral"]
}

helpDict = {
    'anger': 0, 'disgust': 1, 'fear': 2, 'joy': 3,
    'sadness': 4, 'surprise': 5, 'neutral': 6
}

newData = []
for text, emotion in zip(data['Phrase'], data['Emotion']):
#     if len(emotion.split(',')) == 1:
    emotion_ = emotion.split(',')[0]

    for key in ekmanDictMo.keys():
        if emotionDict[emotion_] in ekmanDictMo[key]:
            newData.append([text, key])

df = pd.DataFrame(newData, columns=['text', 'target'])

In [4]:
data = pd.read_table("../input/project-2-data/GoEmotions/train.tsv")
data.columns = ['Phrase', 'Emotion', 'Garbage']
data.drop(['Garbage'], axis=1, inplace=True)

newCol = []
for emotion in data['Emotion']:
    newCol.append(emotionDict[str(emotion.split(',')[0])])
data['Emotion'] = newCol

train_df = data.rename({'Phrase': 'text', 'Emotion': 'target'}, axis=1)

idxLabels = []
for emo in train_df['target']:
    idxLabels.append(emos.index(emo))
    
train_df['label'] = idxLabels

In [12]:
data = pd.read_table("../input/project-2-data/GoEmotions/test.tsv")
data.columns = ['Phrase', 'Emotion', 'Garbage']
data.drop(['Garbage'], axis=1, inplace=True)

newCol = []
for emotion in data['Emotion']:
    newCol.append(emotionDict[str(emotion.split(',')[0])])
data['Emotion'] = newCol

test_df = data.rename({'Phrase': 'text', 'Emotion': 'target'}, axis=1)

idxLabels = []
for emo in test_df['target']:
    idxLabels.append(emos.index(emo))
    
test_df['label'] = idxLabels

In [13]:
data = pd.read_table("../input/project-2-data/GoEmotions/dev.tsv")
data.columns = ['Phrase', 'Emotion', 'Garbage']
data.drop(['Garbage'], axis=1, inplace=True)

newCol = []
for emotion in data['Emotion']:
    newCol.append(emotionDict[str(emotion.split(',')[0])])
data['Emotion'] = newCol

dev_df = data.rename({'Phrase': 'text', 'Emotion': 'target'}, axis=1)

idxLabels = []
for emo in dev_df['target']:
    idxLabels.append(emos.index(emo))
    
dev_df['label'] = idxLabels

In [14]:
train_df = pd.concat([train_df, dev_df], ignore_index=True)

## 2.3 Pre-processing Tweets


In [15]:
from nltk.tokenize import WordPunctTokenizer
import re
import emoji
from bs4 import BeautifulSoup
import itertools

#Emoji patterns
emoji_pattern = re.compile("["
         u"\U0001F600-\U0001F64F"  # emoticons
         u"\U0001F300-\U0001F5FF"  # symbols & pictographs
         u"\U0001F680-\U0001F6FF"  # transport & map symbols
         u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
         u"\U00002702-\U000027B0"
         u"\U000024C2-\U0001F251"
         "]+", flags=re.UNICODE)

tok = WordPunctTokenizer()
pat1 = r'@[A-Za-z0-9]+'
pat2 = r'https?://[A-Za-z0-9./]+'

def tweet_cleaner(text): 
    try:
        text1 = text.decode("utf-8-sig").replace(u"\ufffd", "?") # The UTF-8 BOM is a sequence of bytes (EF BB BF) that allows the reader to identify a file as being encoded in UTF-8
    except:
        text1 = text
    
    
    #replace consecutive non-ASCII characters with a space
    text1 = re.sub(r'[^\x00-\x7F]+',' ', text1)
    
    #remove emojis from tweet
    text2 = emoji_pattern.sub(r'', text1)
    
    # Remove emoticons
    # text3 = [word for word in text2.split() if word not in emoticons]
    # text3 = " ".join(text3)
    
    # contradictions and special characters 
    # text4 = spl_ch_contra(text3)
    
    # HTML encoding
    soup = BeautifulSoup(text2, 'lxml') #HTML encoding has not been converted to text, and ended up in text field as ‘&amp’,’&quot’,etc.
    text5 = soup.get_text()
    
    # removing @ mentions
    text6 = re.sub(pat1, '', text5)
    
    # Removing URLs
    text7 = re.sub(pat2, '', text6)
    
    # Removing punctuations
    # text8 = re.sub("[\.\,\!\?\:\;\-\=\(\)\[\]\"\'\%\*\#\@]", " ", text7)
    
    # Fix misspelled words
    text9 = ''.join(''.join(s)[:2] for _, s in itertools.groupby(text7))# checking that each character should occur not more than 2 times in every word

    # Tokenizing ,change cases & join together to remove unneccessary white spaces
    text9_list = tok.tokenize(text9.lower())
    text10 = (" ".join(text9_list)).strip()
    
    return text10

In [16]:
train_df['text_cleaned'] = list(map(lambda x:tweet_cleaner(x),train_df['text']) )
test_df['text_cleaned'] = list(map(lambda x:tweet_cleaner(x),test_df['text']) )

In [17]:
# Get the lists of sentences and their labels.
train_sentences = train_df.text_cleaned.values
train_labels = train_df.label.values

test_sentences = test_df.text_cleaned.values
test_labels = test_df.label.values

# 3. Tokenization & Input Formatting

In this section, we'll transform our dataset into the format that BERT can be trained on.

In [18]:
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)

In [19]:
def tokenData(sentences, labels):
    input_ids = []
    attention_masks = []

    # For every sentence...
    for sent in sentences:
        encoded_dict = tokenizer.encode_plus(sent, add_special_tokens = True,
                            max_length = 256, pad_to_max_length = True,
                            return_attention_mask = True, return_tensors = 'pt')

        input_ids.append(encoded_dict['input_ids'])    
        attention_masks.append(encoded_dict['attention_mask'])

    # Convert the lists into tensors.
    input_ids = torch.cat(input_ids, dim=0)
    attention_masks = torch.cat(attention_masks, dim=0)
    labels = torch.tensor(labels)

    return input_ids, attention_masks, labels

train_input_ids, train_attention_masks, train_labels = tokenData(train_sentences, train_labels)
test_input_ids, test_attention_masks, test_labels = tokenData(test_sentences, test_labels)

In [20]:
from torch.utils.data import TensorDataset

train_dataset = TensorDataset(train_input_ids, train_attention_masks, train_labels)
val_dataset = TensorDataset(test_input_ids, test_attention_masks, test_labels)

In [21]:
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

batch_size = 16
train_dataloader = DataLoader(
            train_dataset,  # The training samples.
            sampler = RandomSampler(train_dataset), # Select batches randomly
            batch_size = batch_size # Trains with this batch size.
        )

validation_dataloader = DataLoader(
            val_dataset, # The validation samples.
            sampler = SequentialSampler(val_dataset), # Pull out batches sequentially.
            batch_size = batch_size # Evaluate with this batch size.
        )

# 4. Train Our Classification Model

In [28]:
from transformers import BertForSequenceClassification, AdamW, BertConfig

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels = 28, 
    output_attentions = False, output_hidden_states = False, return_dict=False)

model.cuda()

## 4.2. Optimizer & Learning Rate Scheduler

In [30]:
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr = 5e-5, eps = 1e-8)

epochs = 2
total_steps = len(train_dataloader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, 
                                            num_warmup_steps = 0, # Default value in run_glue.py
                                            num_training_steps = total_steps)

In [31]:
import numpy as np

# Function to calculate the accuracy of our predictions vs labels
def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

In [32]:
import time
import datetime

def format_time(elapsed):
    '''
    Takes a time in seconds and returns a string hh:mm:ss
    '''
    # Round to the nearest second.
    elapsed_rounded = int(round((elapsed)))
    
    # Format as hh:mm:ss
    return str(datetime.timedelta(seconds=elapsed_rounded))


In [33]:
import random
import numpy as np

act = []
pred = []

# Set the seed value all over the place to make this reproducible.
seed_val = 66

random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

# We'll store a number of quantities such as training and validation loss, 
# validation accuracy, and timings.
training_stats = []

# Measure the total training time for the whole run.
total_t0 = time.time()

# For each epoch...
for epoch_i in range(0, epochs):
    
    # ========================================
    #               Training
    # ========================================
    
    # Perform one full pass over the training set.

    print("")
    print('======== Epoch {:} / {:} ========'.format(epoch_i + 1, epochs))
    print('Training...')

    # Measure how long the training epoch takes.
    t0 = time.time()

    total_train_loss = 0
    model.train()

    # For each batch of training data...
    for step, batch in enumerate(train_dataloader):

        # Progress update every 40 batches.
        if step % 40 == 0 and not step == 0:
            # Calculate elapsed time in minutes.
            elapsed = format_time(time.time() - t0)
            
            # Report progress.
            print('  Batch {:>5,}  of  {:>5,}.    Elapsed: {:}.'.format(step, len(train_dataloader), elapsed))

        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)

        model.zero_grad()        

        loss, logits = model(b_input_ids, 
                             token_type_ids=None, 
                             attention_mask=b_input_mask, 
                             labels=b_labels)
        total_train_loss += loss.item()

        # Perform a backward pass to calculate the gradients.
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # Update the learning rate.
        scheduler.step()

    # Calculate the average loss over all of the batches.
    avg_train_loss = total_train_loss / len(train_dataloader)            
    
    # Measure how long this epoch took.
    training_time = format_time(time.time() - t0)

    print("")
    print("  Average training loss: {0:.2f}".format(avg_train_loss))
    print("  Training epcoh took: {:}".format(training_time))
        
    # ========================================
    #               Validation
    # ========================================
    # After the completion of each training epoch, measure our performance on
    # our validation set.

    print("")
    print("Running Validation...")

    t0 = time.time()

    # Put the model in evaluation mode--the dropout layers behave differently
    # during evaluation.
    model.eval()

    # Tracking variables 
    total_eval_accuracy = 0
    total_eval_loss = 0
    nb_eval_steps = 0

    # Evaluate data for one epoch
    for batch in validation_dataloader:
        
        b_input_ids = batch[0].to(device)
        b_input_mask = batch[1].to(device)
        b_labels = batch[2].to(device)
        
        with torch.no_grad():        
            (loss, logits) = model(b_input_ids, 
                                   token_type_ids=None, 
                                   attention_mask=b_input_mask,
                                   labels=b_labels)
            
        # Accumulate the validation loss.
        total_eval_loss += loss.item()

        # Move logits and labels to CPU
        logits = logits.detach().cpu().numpy()
        label_ids = b_labels.to('cpu').numpy()
        
        pred_flat = np.argmax(logits, axis=1).flatten()
        labels_flat = label_ids.flatten()
        
        act.append(labels_flat)
        pred.append(pred_flat)

        # Calculate the accuracy for this batch of test sentences, and
        # accumulate it over all batches.
        total_eval_accuracy += flat_accuracy(logits, label_ids)
        

    # Report the final accuracy for this validation run.
    avg_val_accuracy = total_eval_accuracy / len(validation_dataloader)
    print("  Accuracy: {0:.2f}".format(avg_val_accuracy))

    # Calculate the average loss over all of the batches.
    avg_val_loss = total_eval_loss / len(validation_dataloader)
    
    # Measure how long the validation run took.
    validation_time = format_time(time.time() - t0)
    
    print("  Validation Loss: {0:.2f}".format(avg_val_loss))
    print("  Validation took: {:}".format(validation_time))

    # Record all statistics from this epoch.
    training_stats.append(
        {
            'epoch': epoch_i + 1,
            'Training Loss': avg_train_loss,
            'Valid. Loss': avg_val_loss,
            'Valid. Accur.': avg_val_accuracy,
            'Training Time': training_time,
            'Validation Time': validation_time
        }
    )

print("")
print("Training complete!")

print("Total training took {:} (h:mm:ss)".format(format_time(time.time()-total_t0)))

Let's view the summary of the training process.

In [41]:
act_ = np.concatenate(act)
pred_ = np.concatenate(pred)

_Dict = {}
for index, emo in enumerate(emos):
    _Dict[emo] = index

from sklearn.metrics import classification_report
print(classification_report(act_, pred_, target_names=_Dict.keys(), digits=len(_Dict)))